# 🍳 Lab 8: CookSense AI Challenge

**Challenge**: Given a list of ingredients, predict what cuisine the dish belongs to.

**Kaggle competition**: [cook-sense-ai-challenge](https://www.kaggle.com/competitions/cook-sense-ai-challenge)

**What you'll use**: pandas, TF-IDF, Logistic Regression, multi-class classification

---

### The Story
CookSense has a database of recipes. Each recipe has a list of ingredients but the cuisine label is missing for new recipes. Your job: train a model that looks at ingredients and predicts the cuisine.

**Input**: `["soy sauce", "ginger", "sesame oil", "tofu"]`  
**Output**: `"japanese"`

This is a **text classification** problem — your first real NLP challenge!

In [ ]:
# Install dependencies
!pip install pandas scikit-learn matplotlib seaborn kaggle -q

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('✅ Libraries loaded!')

## Step 1: Download the Data from Kaggle

You need a Kaggle account and API key. Follow these steps:

1. Go to [kaggle.com](https://www.kaggle.com) → Your Profile → Settings → API → **Create New Token**
2. This downloads a `kaggle.json` file
3. Upload it to this notebook environment
4. Run the cell below

In [ ]:
# Set up Kaggle credentials
import os

# If running on Google Colab, upload your kaggle.json first
# from google.colab import files
# files.upload()  # upload kaggle.json

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.system('cp kaggle.json ~/.kaggle/')
os.system('chmod 600 ~/.kaggle/kaggle.json')

# Download competition data
os.system('kaggle competitions download -c cook-sense-ai-challenge')
os.system('unzip -o cook-sense-ai-challenge.zip')

print('✅ Data downloaded!')

In [ ]:
# Load the data
with open('train.json', 'r') as f:
    train_data = json.load(f)

with open('test.json', 'r') as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

print(f'Training recipes: {len(train_df):,}')
print(f'Test recipes:     {len(test_df):,}')
print(f'Cuisines:         {train_df["cuisine"].nunique()}')
print()
print('Sample recipe:')
print(train_df.iloc[0])

## Step 2: Explore the Data

Before building any model, always understand your data first.

In [ ]:
# How many recipes per cuisine?
cuisine_counts = train_df['cuisine'].value_counts()

plt.figure(figsize=(14, 5))
cuisine_counts.plot(kind='bar', color='#00ff9d', edgecolor='black')
plt.title('Number of Recipes per Cuisine', fontsize=14)
plt.xlabel('Cuisine')
plt.ylabel('Number of Recipes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Cuisine counts:')
print(cuisine_counts)

In [ ]:
# How many ingredients per recipe?
train_df['num_ingredients'] = train_df['ingredients'].apply(len)

plt.figure(figsize=(10, 4))
train_df['num_ingredients'].hist(bins=30, color='#00ff9d', edgecolor='black')
plt.title('Distribution of Ingredients per Recipe')
plt.xlabel('Number of Ingredients')
plt.ylabel('Number of Recipes')
plt.show()

print(f"Average ingredients per recipe: {train_df['num_ingredients'].mean():.1f}")
print(f"Max: {train_df['num_ingredients'].max()}")
print(f"Min: {train_df['num_ingredients'].min()}")

In [ ]:
# What are the most common ingredients overall?
from collections import Counter

all_ingredients = [ing for recipe in train_df['ingredients'] for ing in recipe]
top_ingredients = Counter(all_ingredients).most_common(20)

names, counts = zip(*top_ingredients)
plt.figure(figsize=(12, 5))
plt.barh(names[::-1], counts[::-1], color='#00ff9d')
plt.title('Top 20 Most Common Ingredients')
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()

print('\nNotice: salt, onions, garlic appear everywhere — not very useful for identifying cuisine!')
print('TF-IDF will automatically downweight these common ingredients.')

## Step 3: Prepare Features

TF-IDF expects a **string** as input, not a list.

We join the ingredient list into a single string:
- `["garlic", "olive oil", "tomato"]` → `"garlic olive oil tomato"`

In [ ]:
# YOUR TURN: Join ingredient lists into strings
# Hint: use .apply() with a lambda that does ' '.join(x)

train_df['ingredient_str'] = train_df['ingredients'].apply(___)  # ← fill this in
test_df['ingredient_str'] = test_df['ingredients'].apply(___)    # ← same thing

# Separate features and labels
X_train_raw = train_df['ingredient_str']
y_train = train_df['cuisine']
X_test_raw = test_df['ingredient_str']

print('Sample ingredient string:')
print(X_train_raw.iloc[0])
print('\nCuisine:', y_train.iloc[0])

## Step 4: TF-IDF Vectorisation

**TF-IDF** converts ingredient strings into numerical vectors.

- Common ingredients (salt, water) → low scores (not useful)
- Rare, cuisine-specific ingredients (miso, tahini) → high scores (very useful)

We also use `ngram_range=(1,2)` to capture two-word phrases like `"soy sauce"` and `"olive oil"` as single features.

In [ ]:
# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),  # capture single words AND two-word phrases
    min_df=2             # ignore ingredients appearing in fewer than 2 recipes
)

# YOUR TURN:
# Fit AND transform training data (learns vocabulary + converts to numbers)
X_train = vectorizer.___(X_train_raw)  # ← use fit_transform

# Transform test data ONLY (don't fit — that would be data leakage!)
X_test = vectorizer.___(X_test_raw)    # ← use transform only

print(f'Training matrix shape: {X_train.shape}')
print(f'  → {X_train.shape[0]:,} recipes × {X_train.shape[1]:,} ingredient features')
print(f'Test matrix shape: {X_test.shape}')
print()
print('Vocabulary sample (first 20 features):')
print(list(vectorizer.vocabulary_.keys())[:20])

## Step 5: Train the Model

In [ ]:
# Split training data into train/validation (80/20)
# stratify=y_train ensures each cuisine is proportionally represented in both splits
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f'Training samples:   {X_tr.shape[0]:,}')
print(f'Validation samples: {X_val.shape[0]:,}')

In [ ]:
# YOUR TURN: Create and train a Logistic Regression model
# Hint: LogisticRegression(max_iter=1000, C=5, random_state=42)

model = ___   # ← create the model
___.fit(___, ___)  # ← fit on training data

print('✅ Model trained!')

## Step 6: Evaluate on Validation Set

In [ ]:
# Predict on validation set
val_predictions = model.predict(X_val)

# Overall accuracy
accuracy = accuracy_score(y_val, val_predictions)
print(f'Validation Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)')
print()

# Per-cuisine breakdown
print('Per-cuisine performance:')
print(classification_report(y_val, val_predictions))

In [ ]:
# Confusion matrix — see where the model gets confused
cuisines = sorted(y_train.unique())
cm = confusion_matrix(y_val, val_predictions, labels=cuisines)

plt.figure(figsize=(16, 13))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=cuisines,
    yticklabels=cuisines,
    linewidths=0.5
)
plt.title('Confusion Matrix — Cuisine Prediction', fontsize=14)
plt.ylabel('Actual Cuisine')
plt.xlabel('Predicted Cuisine')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Look for off-diagonal values — those are mistakes.')
print('Which cuisines does the model confuse most often?')

In [ ]:
# Which ingredients are most predictive for each cuisine?
feature_names = vectorizer.get_feature_names_out()

print('Top 10 most predictive ingredients per cuisine:\n')
for i, cuisine in enumerate(model.classes_):
    top_indices = model.coef_[i].argsort()[-10:][::-1]
    top_features = [feature_names[j] for j in top_indices]
    print(f'{cuisine.upper():15} → {" | ".join(top_features)}')

## Step 7: Generate Submission File

In [ ]:
# Retrain on FULL training data before generating predictions
# More data = better model
model.fit(X_train, y_train)
print('✅ Retrained on full dataset')

# Generate predictions for test set
test_predictions = model.predict(X_test)

# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'cuisine': test_predictions
})

# Save to CSV
submission.to_csv('submission.csv', index=False)

print(f'Submission saved! ({len(submission):,} predictions)')
print()
print('Prediction distribution:')
print(submission['cuisine'].value_counts())
print()
print('First 10 predictions:')
print(submission.head(10))

## Step 8: Submit to Kaggle

1. Go to [kaggle.com/competitions/cook-sense-ai-challenge](https://www.kaggle.com/competitions/cook-sense-ai-challenge)
2. Click **Submit Predictions**
3. Upload `submission.csv`
4. See your score on the leaderboard! 🏆

In [ ]:
# Or submit directly via Kaggle API
os.system('kaggle competitions submit -c cook-sense-ai-challenge -f submission.csv -m "Logistic Regression + TF-IDF baseline"')
print('Submitted!')

## Bonus: Try to Improve Your Score

Once you have a baseline, try these improvements:

In [ ]:
# BONUS 1: Try LinearSVC — often better than Logistic Regression for text
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svc = LinearSVC(C=1.0, max_iter=2000)
svc.fit(X_tr, y_tr)
svc_preds = svc.predict(X_val)
svc_acc = accuracy_score(y_val, svc_preds)
print(f'LinearSVC Accuracy: {svc_acc:.4f} ({svc_acc*100:.1f}%)')
print(f'Improvement over Logistic Regression: {(svc_acc - accuracy)*100:+.1f}%')

In [ ]:
# BONUS 2: Clean ingredient names (remove numbers, special characters)
import re

def clean_ingredients(ingredients):
    cleaned = []
    for ing in ingredients:
        # Remove numbers and special characters, lowercase
        ing = re.sub(r'[^a-zA-Z\s]', '', ing).lower().strip()
        if ing:  # skip empty strings
            cleaned.append(ing)
    return ' '.join(cleaned)

# Apply cleaning
X_train_clean = train_df['ingredients'].apply(clean_ingredients)
X_test_clean = test_df['ingredients'].apply(clean_ingredients)

# Refit vectorizer on cleaned data
vec_clean = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_tr_clean = vec_clean.fit_transform(X_train_clean)
X_te_clean = vec_clean.transform(X_test_clean)

# Train and evaluate
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(X_tr_clean, y_train, test_size=0.2, random_state=42, stratify=y_train)
model_clean = LogisticRegression(max_iter=1000, C=5, random_state=42)
model_clean.fit(X_tr2, y_tr2)
clean_acc = accuracy_score(y_val2, model_clean.predict(X_val2))
print(f'Cleaned data accuracy: {clean_acc:.4f} ({clean_acc*100:.1f}%)')

In [ ]:
# ✅ FINAL TEST CELL
assert 'ingredient_str' in train_df.columns, 'ingredient_str column missing from train_df'
assert 'ingredient_str' in test_df.columns, 'ingredient_str column missing from test_df'
assert X_train.shape[0] == len(train_df), 'X_train row count mismatch'
assert X_test.shape[0] == len(test_df), 'X_test row count mismatch'
assert X_train.shape[1] == X_test.shape[1], 'Train and test must have same number of features'
assert hasattr(model, 'predict'), 'model must be trained'
assert accuracy >= 0.75, f'Accuracy should be >= 75%, got {accuracy:.1%}'
import os
assert os.path.exists('submission.csv'), 'submission.csv not found'

print('🎉 All tests passed!')
print(f'Final validation accuracy: {accuracy*100:.1f}%')
print(f'Submission file ready: submission.csv ({len(submission):,} predictions)')
print()
print('🏆 Head to Kaggle and submit your predictions!')
print('   https://www.kaggle.com/competitions/cook-sense-ai-challenge')